<a href="https://colab.research.google.com/github/stauntonjr/local_llm_notebooks/blob/master/Quantize_and_Evaluate_Mistral_Small_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*More details in this article: [Mistral Small 3: An Excellent 24B-Parameter Wide-Shallow LLM](https://kaitchup.substack.com/p/mistral-small-3-an-excellent-24b)*

This notebook shows how to quantize Mistral Small 3 and evaluate the resulting model. I tested this notebook on Google Colab's A100 (40 GB).

# Quantization

## Install

In [ ]:
!pip install auto-round

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 83.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 73.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.7/792.7 kB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 79.9 MB/s eta 0:00:00
  

## 4-bit Quantization with AutoRound

### Instruct Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "mistralai/Mistral-Small-24B-Instruct-2501"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

from auto_round import AutoRound

bits, group_size, sym = 4, 128, True

autoround = AutoRound(model, tokenizer, nsamples=128, iters=256, gradient_accumulate_steps=2, batch_size=4, bits=bits, group_size=group_size, sym=sym)

autoround.quantize()
output_dir = "Mistral-Small-24B-Instruct-2501-AutoRound-GPTQ-4bit"
autoround.save_quantized(output_dir, format='auto_gptq', inplace=True)

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.9k [00:00<?, ?B/s]

model-00001-of-00010.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

model-00002-of-00010.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

model-00003-of-00010.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

2025-02-04 09:39:24 INFO autoround.py L367: caching done
Quantizing model.layers.39: 100%|██████████| 40/40 [2:24:50<00:00, 217.30s/it]2025-02-04 12:04:14 INFO autoround.py L402: quantization tuning time 8752.545511484146
2025-02-04 12:04:14 INFO autoround.py L418: Summary: quantized 280/281 in the model,  ['lm_head'] have not been quantized
Quantizing model.layers.39: 100%|██████████| 40/40 [2:24:50<00:00, 217.27s/it]
2025-02-04 12:04:14 INFO export.py L130: Saving quantized model to autogptq format, this may take a while...
packing lm_head: 100%|██████████| 281/281 [01:28<00:00,  3.17it/s]                         


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(131072, 5120)
    (layers): ModuleList(
      (0-39): 40 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): QuantLinear()
          (k_proj): QuantLinear()
          (v_proj): QuantLinear()
          (o_proj): QuantLinear()
        )
        (mlp): MistralMLP(
          (gate_proj): QuantLinear()
          (up_proj): QuantLinear()
          (down_proj): QuantLinear()
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((5120,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((5120,), eps=1e-05)
    (rotary_emb): MistralRotaryEmbedding()
  )
  (lm_head): Linear(in_features=5120, out_features=131072, bias=False)
)

### Base Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "mistralai/Mistral-Small-24B-Base-2501"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

from auto_round import AutoRound

bits, group_size, sym = 4, 128, True

autoround = AutoRound(model, tokenizer, nsamples=128, iters=256, gradient_accumulate_steps=2, batch_size=4, bits=bits, group_size=group_size, sym=sym)

autoround.quantize()
output_dir = "Mistral-Small-24B-2501-AutoRound-GPTQ-4bit"
autoround.save_quantized(output_dir, format='auto_gptq', inplace=True)

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.9k [00:00<?, ?B/s]

model-00001-of-00010.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

model-00002-of-00010.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

model-00003-of-00010.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

...

2025-02-04 12:32:45 INFO autoround.py L233: using torch.float16 for quantization tuning
2025-02-04 12:32:45 INFO autoround.py L362: start to cache block inputs
2025-02-04 12:32:53,777 INFO config.py L54: PyTorch version 2.4.1+cu124 available.


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

2025-02-04 12:33:54 INFO autoround.py L367: caching done
Quantizing model.layers.39: 100%|██████████| 40/40 [2:25:02<00:00, 217.42s/it]2025-02-04 14:58:56 INFO autoround.py L402: quantization tuning time 8771.361204862595
2025-02-04 14:58:56 INFO autoround.py L418: Summary: quantized 280/281 in the model,  ['lm_head'] have not been quantized
Quantizing model.layers.39: 100%|██████████| 40/40 [2:25:02<00:00, 217.56s/it]
2025-02-04 14:58:56 INFO export.py L130: Saving quantized model to autogptq format, this may take a while...
packing lm_head: 100%|██████████| 281/281 [01:40<00:00,  2.80it/s]                         


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(131072, 5120)
    (layers): ModuleList(
      (0-39): 40 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): QuantLinear()
          (k_proj): QuantLinear()
          (v_proj): QuantLinear()
          (o_proj): QuantLinear()
        )
        (mlp): MistralMLP(
          (gate_proj): QuantLinear()
          (up_proj): QuantLinear()
          (down_proj): QuantLinear()
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((5120,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((5120,), eps=1e-05)
    (rotary_emb): MistralRotaryEmbedding()
  )
  (lm_head): Linear(in_features=5120, out_features=131072, bias=False)
)

# Evaluation

In [ ]:
!pip install --upgrade transformers langdetect immutabledict optimum auto-gptq lm_eval[vllm]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 31.0 kB/s eta 0:00:00a 0:00:07
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 163.4 kB/s eta 0:00:0000:020:29
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.2/264.2 MB 33.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 66.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 64.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 54.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 65.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 77.0 MB/s eta 0:00:0

In [ ]:
  !lm_eval --model hf \
      --model_args pretrained="Mistral-Small-24B-2501-AutoRound-GPTQ-4bit",dtype="float16" \
      --tasks leaderboard_mmlu_pro\
      --device cuda:0 \
      --batch_size auto \
      --num_fewshot 0 \
      --output_path results/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 02-04 15:41:06 __init__.py:183] Automatically detected platform cuda.
2025-02-04:15:41:07,226 INFO     [__main__.py:279] Verbosity set to INFO
2025-02-04:15:41:12,323 INFO     [__main__.py:376] Selected Tasks: ['leaderboard_mmlu_pro']
2025-02-04:15:41:12,327 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-02-04:15:41:12,327 INFO     [evaluator.py:201] Initializing hf model, with arguments: {'pretrained': 'Mistral-Small-24B-2501-AutoRound-GPTQ-4bit', 'dtype': 'float16'}
2025-02-04:15:41:12,377 INFO     [huggingface.py:132] Using device 'cuda:0'
2025-02-04:15:41:12,844 INFO     [huggingface.py:369] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
/usr/local/lib/python3.11/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:410: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.c

In [ ]:
  !lm_eval --model vllm \
      --model_args pretrained="Mistral-Small-24B-2501-AutoRound-GPTQ-4bit",dtype="float16" \
      --tasks leaderboard_ifeval\
      --device cuda:0 \
      --batch_size auto \
      --output_path results/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 02-04 16:53:00 __init__.py:183] Automatically detected platform cuda.
2025-02-04:16:53:00,968 INFO     [__main__.py:279] Verbosity set to INFO
2025-02-04:16:53:05,702 INFO     [__main__.py:376] Selected Tasks: ['leaderboard_ifeval']
2025-02-04:16:53:05,704 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-02-04:16:53:05,704 INFO     [evaluator.py:201] Initializing vllm model, with arguments: {'pretrained': 'Mistral-Small-24B-2501-AutoRound-GPTQ-4bit', 'dtype': 'float16'}
INFO 02-04 16:53:11 config.py:526] This model supports multiple tasks: {'generate', 'classify', 'reward', 'embed', 'score'}. Defaulting to 'generate'.
INFO 02-04 16:53:12 gptq_marlin.py:109] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
WARNING 02-04 16:53:12 config.py:975] MLA is not supported with gptq_marlin quantization. Disabling MLA.
INFO 02-04 16:53:12 llm_engine

In [ ]:
  !lm_eval --model hf \
      --model_args pretrained="Mistral-Small-24B-Instruct-2501-AutoRound-GPTQ-4bit",dtype="float16" \
      --tasks leaderboard_mmlu_pro\
      --device cuda:0 \
      --batch_size auto \
      --num_fewshot 0 \
      --output_path results/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 02-04 16:14:35 __init__.py:183] Automatically detected platform cuda.
2025-02-04:16:14:35,370 INFO     [__main__.py:279] Verbosity set to INFO
2025-02-04:16:14:40,106 INFO     [__main__.py:376] Selected Tasks: ['leaderboard_mmlu_pro']
2025-02-04:16:14:40,108 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-02-04:16:14:40,109 INFO     [evaluator.py:201] Initializing hf model, with arguments: {'pretrained': 'Mistral-Small-24B-Instruct-2501-AutoRound-GPTQ-4bit', 'dtype': 'float16'}
2025-02-04:16:14:40,157 INFO     [huggingface.py:132] Using device 'cuda:0'
2025-02-04:16:14:40,595 INFO     [huggingface.py:369] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
/usr/local/lib/python3.11/dist-packages/auto_gptq/nn_modules/triton_utils/kernels.py:410: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `to

In [ ]:
 !lm_eval --model vllm \
      --model_args pretrained="Mistral-Small-24B-Instruct-2501-AutoRound-GPTQ-4bit",dtype="float16" \
      --tasks leaderboard_ifeval\
      --device cuda:0 \
      --batch_size auto \
      --output_path results/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


INFO 02-04 16:43:34 __init__.py:183] Automatically detected platform cuda.
2025-02-04:16:43:34,681 INFO     [__main__.py:279] Verbosity set to INFO
2025-02-04:16:43:39,409 INFO     [__main__.py:376] Selected Tasks: ['leaderboard_ifeval']
2025-02-04:16:43:39,411 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-02-04:16:43:39,411 INFO     [evaluator.py:201] Initializing vllm model, with arguments: {'pretrained': 'Mistral-Small-24B-Instruct-2501-AutoRound-GPTQ-4bit', 'dtype': 'float16'}
INFO 02-04 16:43:44 config.py:526] This model supports multiple tasks: {'generate', 'score', 'embed', 'reward', 'classify'}. Defaulting to 'generate'.
INFO 02-04 16:43:45 gptq_marlin.py:109] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
WARNING 02-04 16:43:45 config.py:975] MLA is not supported with gptq_marlin quantization. Disabling MLA.
INFO 02-04 16:43:45 l